In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

font_path      = 'font-folder/Roboto-Light.ttf'
font_path_bold = 'font-folder/Roboto-Bold.ttf'
font_prop       = fm.FontProperties(fname=font_path,      size=11)
font_prop_small = fm.FontProperties(fname=font_path,      size=9)
font_prop_title = fm.FontProperties(fname=font_path_bold, size=14)
print('Font setup complete.')

In [ ]:
FILTERED_MODELS = [
    'TimeGPT',
    'TimesFM',
    'Chronos-2',
    'TabPFN-TS',
    'Moirai-2',
    'Lag-Llama',
]

COMMON_MODEL_COLORS = {
    'Chronos-2':   '#4ECED4',
    'Lag-Llama': '#3E977E',
    'Moirai-2':    '#967ECD',
    'TabPFN-TS': '#CE5858',
    'TimeGPT':   '#1F77B4',
    'TimesFM':   '#F2B134',
}

HUB_MODEL_NAMES = {
    'CovidHub': {'baseline': 'CovidHub-baseline', 'ensemble': 'CovidHub-ensemble'},
    'FluSight': {'baseline': 'FluSight-baseline', 'ensemble': 'FluSight-ensemble'},
}

SPECIAL_MODEL_COLORS = {
    'baseline': '#807B6D',
    'ensemble': '#20B2AA',
}

FLU_FILE   = '../model-evaluation/forecast_scores/flusight_all_ranks_2024-2025.csv'
COVID_FILE = '../model-evaluation/forecast_scores/covidhub_all_ranks_2024-2025.csv'
OUTPUT_DIR = '../model-evaluation/figures/top_bottom_maps/'
THRESHOLDS = [0.30]

def get_special_models(hub_name):
    return HUB_MODEL_NAMES[hub_name]['baseline'], HUB_MODEL_NAMES[hub_name]['ensemble']

def get_model_colors(hub_name):
    baseline, ensemble = get_special_models(hub_name)
    colors = COMMON_MODEL_COLORS.copy()
    colors[baseline] = SPECIAL_MODEL_COLORS['baseline']
    colors[ensemble] = SPECIAL_MODEL_COLORS['ensemble']
    return colors

def get_display_name(model, hub_name):
    baseline, ensemble = get_special_models(hub_name)
    if model == baseline: return 'hub-baseline'
    if model == ensemble: return 'hub-ensemble'
    return model

def get_model_marker(model, hub_name):
    baseline, ensemble = get_special_models(hub_name)
    if model == baseline: return 's'
    if model == ensemble: return 'D'
    return 'o'

print('Configuration loaded.')

In [ ]:
def compute_summary(data, metric_col, threshold):
    round_cols = ['reference_date', 'target_end_date', 'horizon', 'location']
    round_cols = [c for c in round_cols if c in data.columns]
    temp = data.dropna(subset=[metric_col]).copy()
    rows = []
    for _, g in temp.groupby(round_cols):
        g = g.copy()
        if len(g) < 2:
            continue
        g = g.sort_values(metric_col, ascending=False).reset_index(drop=True)
        n = len(g)
        k = max(1, math.ceil(threshold * n))
        g['is_top']    = 0
        g['is_bottom'] = 0
        g.loc[n - k:, 'is_top']    = 1
        g.loc[:k - 1, 'is_bottom'] = 1
        rows.append(g[['model', 'is_top', 'is_bottom']])
    if not rows:
        return pd.DataFrame(columns=['model', 'top_pct', 'bottom_pct'])
    res = pd.concat(rows, ignore_index=True)
    return (
        res.groupby('model')
        .agg(
            top_pct    = ('is_top',    lambda x: 100 * x.mean()),
            bottom_pct = ('is_bottom', lambda x: 100 * x.mean()),
        )
        .reset_index()
    )


def plot_panel(ax, summary, panel_title, hub_name, threshold, xlabel=True, ylabel=True):
    """Plot a single top-bottom scatter panel."""
    model_colors = get_model_colors(hub_name)
    baseline_model, ensemble_model = get_special_models(hub_name)
    highlighted = FILTERED_MODELS + [baseline_model, ensemble_model]

    if summary.empty:
        ax.set_title(panel_title, fontproperties=font_prop_title, loc='left', x=0.02)
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center',
                transform=ax.transAxes, fontproperties=font_prop)
        return

    x_split = summary['bottom_pct'].median()
    y_split = summary['top_pct'].median()

    other_df = summary[~summary['model'].isin(highlighted)]
    high_df  = summary[summary['model'].isin(highlighted)]

    # grey background points
    ax.scatter(
        other_df['bottom_pct'], other_df['top_pct'],
        s=40, color='lightgray', edgecolor='gray',
        linewidth=0.4, alpha=0.6, zorder=2, marker='o',
    )

    # highlighted models
    for _, row in high_df.iterrows():
        model = row['model']
        ax.scatter(
            row['bottom_pct'], row['top_pct'],
            s=60,
            color=model_colors.get(model, 'black'),
            edgecolor='white', linewidth=0.6,
            alpha=0.95, zorder=5,
            marker=get_model_marker(model, hub_name),
        )

    # split lines
    ax.axvline(x_split, color='#aaaaaa', linestyle='--', linewidth=0.8, zorder=1)
    ax.axhline(y_split, color='#aaaaaa', linestyle='--', linewidth=0.8, zorder=1)
    ax.axvline(0,       color='#3A373A', linestyle='-.', linewidth=1.0, alpha=0.5, zorder=1)

    ax.set_xlim(-5, 100)
    ax.set_ylim(-5, 100)
    ax.set_xticks(np.arange(0, 101, 20))
    ax.set_yticks(np.arange(0, 101, 20))

    lc = '#3A373A'
    ax.text(1,  96, 'Top performers',          fontproperties=font_prop_small, color=lc, ha='left',  va='top')
    ax.text(98, 96, 'Inconsistent performers', fontproperties=font_prop_small, color=lc, ha='right', va='top')
    ax.text(1,  1,  'Consistent performers',   fontproperties=font_prop_small, color=lc, ha='left',  va='bottom')
    ax.text(98, 1,  'Frequent low performers', fontproperties=font_prop_small, color=lc, ha='right', va='bottom')

    ax.set_title(panel_title, fontproperties=font_prop_title, loc='left', x=0.02, pad=10)
    if xlabel:
        ax.set_xlabel(f'% of rounds in the bottom {int(threshold * 100)}%', fontproperties=font_prop)
    if ylabel:
        ax.set_ylabel(f'% of rounds in the top {int(threshold * 100)}%', fontproperties=font_prop)

    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(font_prop_small)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.grid(alpha=0.18, linewidth=0.4)


print('Functions defined.')

In [ ]:
# ============================================================
# MAIN — one 2×2 figure per threshold
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_flu   = pd.read_csv(FLU_FILE)
df_covid = pd.read_csv(COVID_FILE)

wis_col = 'wis'
ae_col  = 'ae'

# build legend handles once (same for all thresholds)
flu_colors   = get_model_colors('FluSight')
highlighted  = FILTERED_MODELS + [get_special_models('FluSight')[0], get_special_models('FluSight')[1]]
legend_handles = [
    Line2D([0], [0],
           marker=get_model_marker(m, 'FluSight'), color='w',
           label=get_display_name(m, 'FluSight'),
           markerfacecolor=flu_colors.get(m, 'black'),
           markeredgecolor='white', markersize=7, linewidth=0)
    for m in highlighted
]
legend_handles.append(
    Line2D([0], [0], marker='o', color='w', label='Other models',
           markerfacecolor='lightgray', markeredgecolor='gray',
           markersize=7, linewidth=0)
)

for threshold in THRESHOLDS:
    flu_wis   = compute_summary(df_flu,   wis_col, threshold)
    flu_ae    = compute_summary(df_flu,   ae_col,  threshold)
    covid_wis = compute_summary(df_covid, wis_col, threshold)
    covid_ae  = compute_summary(df_covid, ae_col,  threshold)

    fig = plt.figure(figsize=(16, 10), dpi=300)
    gs  = GridSpec(
        2, 2,
        wspace=0.30,
        hspace=0.45,
        left=0.06, right=0.86,
        top=0.94,  bottom=0.08
    )

    ax_A = fig.add_subplot(gs[0, 0])
    ax_B = fig.add_subplot(gs[0, 1])
    ax_C = fig.add_subplot(gs[1, 0])
    ax_D = fig.add_subplot(gs[1, 1])

    plot_panel(ax_A, flu_wis,   f'A.\u2003FluSight \u2013 WIS',   'FluSight', threshold, xlabel=False)
    plot_panel(ax_B, flu_ae,    f'B.\u2003FluSight \u2013 AE',    'FluSight', threshold, xlabel=False, ylabel=False)
    plot_panel(ax_C, covid_wis, f'C.\u2003CovidHub \u2013 WIS',   'CovidHub', threshold)
    plot_panel(ax_D, covid_ae,  f'D.\u2003CovidHub \u2013 AE',    'CovidHub', threshold, ylabel=False)

    fig.legend(
        handles=legend_handles,
        loc='center left',
        bbox_to_anchor=(0.87, 0.5),
        frameon=False,
        prop=font_prop,
    )

    pct = int(threshold * 100)
    base = os.path.join(OUTPUT_DIR, f'top_bottom_map_{pct}pct_2024-2025')
    fig.savefig(base + '.svg', bbox_inches='tight')
    fig.savefig(base + '.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Saved: {base}.svg / .png')